In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interactive_output, FloatSlider, HBox, VBox, Layout, HTML, Output
from IPython.display import display

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Explore the step response of a second-order series RLC low-pass filter as the circuit parameters R, L, and C are varied.</div>
<div><b>What we see:</b> The output voltage v₀(t) produced by a unit-step input, together with the values of the natural frequency ω₀, quality factor Q, and damping factor ζ.</div>
<div><b>What happens as we interact:</b> Changing R, L, and C modifies Q and ζ and therefore changes the transient behavior of the circuit. The response may be underdamped, critically damped, or overdamped.</div>
</div>
""")

# ------------------------------------------------------------
# 2. CONTROLS
# ------------------------------------------------------------

r_slider = FloatSlider(min=1.0, max=100.0, step=1.0, value=20.0, description='R:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'30px'}, layout=Layout(width='220px'))

l_slider = FloatSlider(min=1.0, max=100.0, step=1.0, value=50.0, description='L:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'30px'}, layout=Layout(width='220px'))

c_slider = FloatSlider(min=1.0, max=200.0, step=1.0, value=100.0, description='C:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'30px'}, layout=Layout(width='220px'))

# ------------------------------------------------------------
# 3. CONTROL INFORMATION
# ------------------------------------------------------------

parameter_label = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:4px;
    margin-bottom:5px;
">
Circuit Parameters:
</div>
""")

units_html = HTML("""
<div style="
    font-size:12px;
    line-height:1.5;
    margin-top:7px;
    color:#555555;
">
R in Ω<br>
L in mH<br>
C in μF
</div>
""")

# ------------------------------------------------------------
# 4. CUSTOM LEGEND
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:135px;
    font-size:13px;
    line-height:1.7;
    background:white;
">

<div>
<span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>
v₀(t)
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dashed #888888; vertical-align:middle; margin-right:7px;"></span>
Unit step
</div>

</div>
""")

# ------------------------------------------------------------
# 5. OUTPUT WIDGETS
# ------------------------------------------------------------

plot_output = Output(layout=Layout(width='calc(100% - 230px)', overflow='visible'))

info_output = Output(layout=Layout(width='auto', overflow='visible'))

# ------------------------------------------------------------
# 6. MAIN FUNCTION
# ------------------------------------------------------------

def plot_rlc_step(R, L, C):

    # Convert mH and μF to SI units
    L_si = L * 1e-3
    C_si = C * 1e-6

    # --------------------------------------------------------
    # CIRCUIT PARAMETERS
    # --------------------------------------------------------

    omega0 = 1.0 / np.sqrt(L_si * C_si)

    Q = np.sqrt(L_si / (C_si * R**2))

    zeta = 1.0 / (2.0 * Q)

    # --------------------------------------------------------
    # DAMPING REGIME
    # --------------------------------------------------------

    tolerance = 0.02

    if zeta < 1.0 - tolerance:
        regime = 'Underdamped'
    elif zeta > 1.0 + tolerance:
        regime = 'Overdamped'
    else:
        regime = 'Critically damped'

    # --------------------------------------------------------
    # TRANSFER FUNCTION
    #
    # H(s) = 1 / (LC s² + RC s + 1)
    # --------------------------------------------------------

    numerator = [1.0]

    denominator = [L_si * C_si, R * C_si, 1.0]

    system = signal.TransferFunction(numerator, denominator)

    # --------------------------------------------------------
    # AUTOMATIC TIME INTERVAL
    # --------------------------------------------------------

    poles = np.roots(denominator)

    negative_real_parts = np.abs(np.real(poles[np.real(poles) < 0]))

    if len(negative_real_parts) > 0:
        slowest_decay = np.min(negative_real_parts)
        t_end = 7.0 / slowest_decay
    else:
        t_end = 10.0 * 2.0 * np.pi / omega0

    natural_period = 2.0 * np.pi / omega0

    t_end = max(t_end, 4.0 * natural_period)

    t = np.linspace(0.0, t_end, 2000)

    # --------------------------------------------------------
    # STEP RESPONSE
    # --------------------------------------------------------

    t, vout = signal.step(system, T=t)

    # --------------------------------------------------------
    # DISPLAY TIME UNIT
    # --------------------------------------------------------

    if t_end < 0.01:
        time_scale = 1e6
        time_unit = 'μs'
    elif t_end < 1.0:
        time_scale = 1e3
        time_unit = 'ms'
    else:
        time_scale = 1.0
        time_unit = 's'

    t_display = t * time_scale

    # --------------------------------------------------------
    # RESPONSE CHARACTERISTICS
    # --------------------------------------------------------

    peak_value = np.max(vout)

    peak_index = np.argmax(vout)

    peak_time = t_display[peak_index]

    overshoot = max(0.0, (peak_value - 1.0) * 100.0)

    # --------------------------------------------------------
    # 7. PLOT
    # --------------------------------------------------------

    with plot_output:

        plot_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 4.6))

        ax.plot(t_display, vout, 'r-', linewidth=2.0)

        ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.2)

        if regime == 'Underdamped':
            ax.plot(peak_time, peak_value, 'ko', markersize=4)

        ymin = min(0.0, np.min(vout) - 0.1)

        ymax = max(1.4, np.max(vout) + 0.15)

        ax.set_xlim(0.0, t_display[-1])

        ax.set_ylim(ymin, ymax)

        ax.set_xlabel(f'Time t ({time_unit})', fontsize=11)

        ax.set_ylabel('Output Voltage v₀(t)', fontsize=11)

        ax.set_title('Step Response of a Series RLC Low-Pass Filter', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, linestyle=':', alpha=0.35)

        plt.tight_layout()

        plt.show()

        plt.close(fig)

    # --------------------------------------------------------
    # 8. INFORMATION FRAME
    # --------------------------------------------------------

    if regime == 'Underdamped':
        transient_text = f'<b>Peak:</b> <span style="color:#0066cc;">{peak_value:.3f}</span>&nbsp;&nbsp;&nbsp;<b>Overshoot:</b> <span style="color:#0066cc;">{overshoot:.1f}%</span>'
    else:
        transient_text = '<b>Overshoot:</b> <span style="color:#0066cc;">0%</span>'

    info_html = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:7px 11px;
        font-size:12.5px;
        background:white;
        width:fit-content;
    ">

        <div style="
            display:flex;
            flex-direction:row;
            align-items:center;
            justify-content:flex-start;
            gap:20px;
            white-space:nowrap;
        ">

            <div>
                <b>R:</b>
                <span style="color:#0066cc;">{R:.1f} Ω</span>
            </div>

            <div>
                <b>L:</b>
                <span style="color:#0066cc;">{L:.1f} mH</span>
            </div>

            <div>
                <b>C:</b>
                <span style="color:#0066cc;">{C:.1f} μF</span>
            </div>

            <div>
                <b>ω₀:</b>
                <span style="color:#0066cc;">{omega0:.2f} rad/s</span>
            </div>

            <div>
                <b>Q:</b>
                <span style="color:#0066cc;">{Q:.3f}</span>
            </div>

            <div>
                <b>ζ:</b>
                <span style="color:#0066cc;">{zeta:.3f}</span>
            </div>

        </div>

        <div style="
            margin-top:5px;
            padding-top:5px;
            border-top:1px solid #eeeeee;
            white-space:nowrap;
        ">

            <b>Damping:</b>
            <span style="color:#0066cc;">{regime}</span>

            <span style="margin-left:22px;">
                {transient_text}
            </span>

        </div>

    </div>
    """

    with info_output:

        info_output.clear_output(wait=True)

        display(HTML(info_html))

# ------------------------------------------------------------
# 9. INTERACTION
# ------------------------------------------------------------

interactive_controls = interactive_output(plot_rlc_step, {'R': r_slider, 'L': l_slider, 'C': c_slider})

interactive_controls.layout.display = 'none'

# ------------------------------------------------------------
# 10. LEFT CONTROL COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, parameter_label, r_slider, l_slider, c_slider, units_html], layout=Layout(width='230px', min_width='230px', align_items='flex-start', padding='0px 0px 0px 4px'))

# ------------------------------------------------------------
# 11. MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, plot_output], layout=Layout(width='100%', align_items='flex-start', justify_content='flex-start', overflow='visible'))

# ------------------------------------------------------------
# 12. INFORMATION AREA
# ------------------------------------------------------------

info_spacer = HTML("", layout=Layout(width='230px', min_width='230px'))

info_area = HBox([info_spacer, info_output], layout=Layout(width='100%', align_items='flex-start', justify_content='flex-start', overflow='visible'))

# ------------------------------------------------------------
# 13. FINAL DISPLAY
# ------------------------------------------------------------

display(description)

display(main_area)

display(info_area)

display(interactive_controls)